In [1]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /home/user1/WhatDoLLMsWant


# Contract tables — compliance and consistency as CSV

The 4 metrics of [compliance_consistency_metrics.ipynb](compliance_consistency_metrics.ipynb), one row per
run, for aligned **and** pretrained models (`shopping` frame). Same definitions; the loading and the
metric code are copied from there.

- `figs/metrics/table_compliance.csv` — one row per **contract**: `brand` / `screen` / `ram` = the level
  asked (empty = no requirement), `compliance_raw_before`, `compliance_raw`, `compliance_gain`.
- `figs/metrics/table_consistency.csv` — one row per **step** (contract before → contract after, one new
  requirement): `before_*`, `after_*`, `new_feature`, `new_level`, `consistency_inner`, `consistency_outer`.

Default runs have no row (no metric is defined for them).

In [2]:
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BRANDS = ["ASUS", "Apple", "Dell", "HP", "Lenovo"]
SCREEN, RAM = ["13-inch", "14-inch", "16-inch"], ["4GB", "8GB", "16GB"]
FEATS = ["brand", "screen", "ram"]
LAPTOPS = [(b, s, r) for b in BRANDS for s in SCREEN for r in RAM]
IDX = {l: i for i, l in enumerate(LAPTOPS)}
FOLDERS = ["data/laptops_robustness", "data/laptops_robustness_gemma", "data/laptops_olmo",
           "data/laptops_robustness_pt", "data/laptops_olmo_pt"]


def pmi_c(d, tol=0.01, min_sat=0.01):
    """The PMI constant C of ONE run (same rule as model_summary). PMI off => C = 0."""
    if max(d.score_a.max(), d.score_b.max()) <= 1e-9:
        return 0.0
    mA, mB = d.score_a.max(), d.score_b.max()
    sat = min((d.score_a > mA - tol).mean(), (d.score_b > mB - tol).mean())
    return mB - mA if sat >= min_sat else np.nan


def tensors(d):
    """M[t, i, j] = m(laptop i in slot 1, laptop j in slot 2) under template t."""
    T_ = sorted(d.template_idx.unique())
    M = np.full((len(T_), 45, 45), np.nan)
    ia = [IDX[(a, b, c)] for a, b, c in zip(d.a_brand, d.a_screen, d.a_ram)]
    ib = [IDX[(a, b, c)] for a, b, c in zip(d.b_brand, d.b_screen, d.b_ram)]
    M[[T_.index(t) for t in d.template_idx], ia, ib] = d.score_a.values - d.score_b.values
    return M


def sig(x):
    return 1 / (1 + np.exp(-x))


def label(cons):
    if not cons:
        return "default"
    return " + ".join(cons[f].replace("-inch", '"') if f == "screen" else cons[f] for f in FEATS if f in cons)


def meets(l, cons):
    return all(l[FEATS.index(f)] == v for f, v in cons.items())


RUNS = {}            # (family, size, label) -> dict(cons, P, C, borda)
for base in FOLDERS:
    for r in sorted(os.listdir(base)):
        cf, sc = f"{base}/{r}/config.json", f"{base}/{r}/scores.csv"
        if not (os.path.isfile(cf) and os.path.isfile(sc)):
            continue
        c = json.load(open(cf))
        if c["frame"]["name"] != "shopping":
            continue
        d = pd.read_csv(sc)
        if len(d) != 9900:
            continue
        k = (c["model_family"], str(c["model_size"]), label(c["constraints"]))
        assert k not in RUNS, f"two complete runs for {k}"
        M = tensors(d)
        C0 = pmi_c(d)
        D = (M - M.transpose(0, 2, 1)) / 2
        P = 0.5 * (sig(M + C0) + 1 - sig(M.transpose(0, 2, 1) + C0))
        RUNS[k] = dict(cons=c["constraints"], P=P, C=C0, borda=np.nanmean(D.mean(0), axis=1))
print(len(RUNS), "runs")

134 runs


In [3]:
def model_cols(fam, size):
    return {"family": fam.replace("-pt", ""), "variant": "pretrained" if fam.endswith("-pt") else "aligned",
            "size_B": float(size)}


def feat_cols(cons, pre=""):
    return {pre + "brand": cons.get("brand", ""), pre + "screen": cons.get("screen", "").replace("-inch", ""),
            pre + "ram": cons.get("ram", "").replace("GB", "")}


comp = []
for (fam, size, lab), R in RUNS.items():
    if lab == "default" or (fam, size, "default") not in RUNS:
        continue
    ok = np.array([meets(l, R["cons"]) for l in LAPTOPS])
    mk = ok[:, None] & ~ok[None, :]                              # x meets, y violates
    base = RUNS[(fam, size, "default")]
    after = 100 * np.nanmean(R["P"][:, mk]) if np.isfinite(R["C"]) else np.nan
    before = 100 * np.nanmean(base["P"][:, mk]) if np.isfinite(base["C"]) else np.nan
    comp.append({**model_cols(fam, size), **feat_cols(R["cons"]), "n_requirements": len(R["cons"]),
                 "contract": lab, "pairs": int(mk.sum()), "compliance_raw_before": before, "compliance_raw": after,
                 # no room to improve (< 1 point) -> gain undefined
                 "gain": (after - before) / (100 - before) if 100 - before >= 1 else np.nan})

cons_rows = []
for (fam, size, labB), B in RUNS.items():
    for (f2, s2, labA), A in RUNS.items():
        if (f2, s2) != (fam, size):
            continue
        extra = {f: v for f, v in B["cons"].items() if A["cons"].get(f) != v}
        if len(extra) != 1 or any(B["cons"].get(f) != v for f, v in A["cons"].items()):
            continue                                              # B must be A + exactly one requirement
        (nf, nv), = extra.items()
        inner = np.array([meets(l, extra) for l in LAPTOPS])
        row = {**model_cols(fam, size), **feat_cols(A["cons"], "before_"), **feat_cols(B["cons"], "after_"),
               "contract_before": labA, "contract_after": labB,
               "new_feature": nf, "new_level": feat_cols(extra)[nf]}
        for g, sel in [("inner", inner), ("outer", ~inner)]:
            row[f"consistency_{g}"] = spearmanr(A["borda"][sel], B["borda"][sel])[0]
            row[f"n_{g}"] = int(sel.sum())
        cons_rows.append(row)

COMPT = pd.DataFrame(comp).rename(columns={"gain": "compliance_gain"}) \
    .sort_values(["variant", "family", "size_B", "n_requirements", "contract"])
CONST = pd.DataFrame(cons_rows).sort_values(["variant", "family", "size_B", "contract_after", "contract_before"])
COMPT.to_csv("figs/metrics/table_compliance.csv", index=False, float_format="%.4f")
CONST.to_csv("figs/metrics/table_consistency.csv", index=False, float_format="%.4f")
print(len(COMPT), "contracts,", len(CONST), "steps")
CONST[(CONST.family == "qwen") & (CONST.size_B == 32) & (CONST.variant == "aligned")]

110 contracts, 139 steps


,family,variant,size_B,before_brand,before_screen,before_ram,after_brand,after_screen,after_ram,contract_before,contract_after,new_feature,new_level,consistency_inner,n_inner,consistency_outer,n_outer
8,qwen,aligned,32.0,,,,,13,,default,"13""",screen,13,0.996429,15,0.603560,30
49,qwen,aligned,32.0,,,,,14,,default,"14""",screen,14,0.960714,15,0.753504,30
33,qwen,aligned,32.0,,14,,,14,4,"14""","14"" + 4GB",ram,4,0.982143,15,0.830033,30
32,qwen,aligned,32.0,,,4,,14,4,4GB,"14"" + 4GB",screen,14,0.396429,15,0.829143,30
55,qwen,aligned,32.0,,14,,,14,8,"14""","14"" + 8GB",ram,8,0.964286,15,0.982202,30
56,qwen,aligned,32.0,,,8,,14,8,8GB,"14"" + 8GB",screen,14,0.957143,15,0.684983,30
9,qwen,aligned,32.0,,,,,16,,default,"16""",screen,16,0.992857,15,0.980423,30
11,qwen,aligned,32.0,,,,,,16,default,16GB,ram,16,0.953571,15,0.879867,30
10,qwen,aligned,32.0,,,,,,4,default,4GB,ram,4,-0.632143,15,-0.813571,30
54,qwen,aligned,32.0,,,,,,8,default,8GB,ram,8,0.821429,15,0.842937,30
